In [2]:
from pyspark.sql import SparkSession
import pyspark

In [4]:
spark = SparkSession.builder \
        .master('local[2]') \
        .appName('homework') \
        .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
26/03/09 16:48:19 WARN Utils: Your hostname, BlackBeast resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 16:48:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 16:48:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.version

'3.5.0'

26/03/09 16:48:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
df_yellow = spark.read \
            .option('header','true') \
            .parquet("./yellow_tripdata_2025-11.parquet")

In [11]:
df_yellow.repartition(4).write.parquet("./pq/",mode='overwrite')

In [13]:
df_yellow.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [17]:
from pyspark.sql import functions as F

In [18]:
df_yellow = df_yellow \
    .withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)) 

In [19]:
df_yellow.filter(df_yellow.pickup_date == '2025-11-15').count()

162604

In [20]:
df_yellow = df_yellow.withColumn(
    'trip_duration_hours', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

In [40]:
df_yellow.orderBy(F.desc("trip_duration_hours")).select('trip_duration_hours').head()

Row(trip_duration_hours=90.64666666666666)

In [25]:
df_zones = spark.read.parquet("../practice/data/zones.parquet")

In [26]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [31]:
df_join = df_yellow.join(df_zones.select("LocationID","Zone"), df_yellow.PULocationID == df_zones.LocationID)

In [33]:
df_join.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+--------------------+----------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date| trip_duration_hours|LocationID|                Zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-

In [38]:
df_join.groupBy('Zone').count().orderBy("count").head(5)

[Row(Zone='Arden Heights', count=1),
 Row(Zone="Eltingville/Annadale/Prince's Bay", count=1),
 Row(Zone="Governor's Island/Ellis Island/Liberty Island", count=1),
 Row(Zone='Port Richmond', count=3),
 Row(Zone='Rikers Island', count=4)]